In [ ]:
import ast
import ipynbname
import pandas as pd
import numpy as np
from Functions.AutoCloud_V2 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *
from Functions.TedaOptimize_QN import *

FileName = ipynbname.name()
out_path = f'Optimization\\{FileName[:-4]}\\multi\\Optimization.csv'
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')
sig = HI['PC1'].values

In [2]:
df = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler=None,OptPrune=False,
                         n_study=1,timeout=60,n_trials=5e1,patience=None,
                         mS=[2.0,4.5],nRS=[1,70],mdS=[1,1],actS=[0,1])

=== ÚLTIMOS 3 TRIALS REGISTRADOS ===
Trial #047 [PRUNED]
Trial #048 [PRUNED]
Trial #049 [PRUNED]
----------------------------------------------------------------------
Melhor Score Geral (Soma): 1.8688 | Sem melhora: 0/12


In [ ]:
for i in range(1,6):
    dfs = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler=None,OptPrune=False,
                            n_study=6,timeout=1680,n_trials=2e4,patience=2e3,
                            mS=[2.0,4.5],nLS=[i,i],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [ ]:
#df = df[(df.iloc[:,0] <= 0.07)]
params_list = df.values[:,-9:]
df

,MAPE_RUL,MAPE_HI,m,nI,nR,nO,mO,N,TAU,past/ahead,activation
128,0.967872,2.278287,2.75,16,"[53, 36, 8, 56]",6,13,7,11.000000,1,0
120,2.170944,0.202906,4.25,20,"[18, 24]",10,19,9,24.000000,1,1
7,2.218315,0.103551,4.50,19,"[29, 41, 65]",9,9,2,2.460463,1,0
4,2.183819,0.126204,2.75,13,"[6, 65, 34, 24, 46]",10,5,1,1.235820,1,1
6,2.111962,0.192931,2.50,7,"[37, 70, 56, 36]",12,6,1,1.977056,1,0
...,...,...,...,...,...,...,...,...,...,...,...
39,0.038132,0.071422,3.50,17,[28],14,5,5,17.000000,1,0
57,0.038132,0.071422,3.25,17,[28],14,5,5,17.000000,1,0
72,0.037961,0.071428,3.75,17,[28],14,5,5,17.000000,1,0
56,0.036604,0.071432,3.50,17,[28],14,5,7,17.000000,1,0


In [ ]:
tedas = []
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1],mode=mode,act=act,
                tau=τ,rho=0.03,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])
    

In [ ]:
PlotDSI_3D_PLT(tedas[0])